In [ ]:
import pandas as pd
import numpy as np

## Merge Contract with Weighted Season and Playoff Stats

We combine the contract data with weighted regular season and playoff statistics.
For each contract year, we compute a weighted average of the stats from the two prior seasons and the contract season:
- weight (year-2): 0.2
- weight (year-1): 0.3
- weight (year):   0.5

The resulting dataframe includes:
- Player, YRS, is_retained, year (contract year)
- age (from regular season stats for the contract year)
- Weighted regular season stats (suffixed `_reg`)
- Weighted playoff stats (suffixed `_playoff`)


In [ ]:
# Load data
contract_df = pd.read_csv('../../data/processed/contract_data.csv')
stats_reg_df = pd.read_csv('../../data/processed/Stats_reg_2014_2024_cleaned.csv')
stats_playoff_df = pd.read_csv('../../data/processed/Stats_playoffs_2014_2024_cleaned.csv')

# Identify numeric statistic columns (exclude identifier columns)
id_cols = ['Player', 'year', 'Team', 'Pos', 'Age']  # Age will be used separately
stat_cols_reg = [c for c in stats_reg_df.columns if c not in id_cols]
stat_cols_playoff = [c for c in stats_playoff_df.columns if c not in id_cols]

# Helper to compute weighted average for a given player and target year
def weighted_stats(df, player, target_year, weights=[0.2, 0.3, 0.5]):
    """Return weighted average of stats for player across target_year-2, target_year-1, target_year."""
    # Determine stat columns based on df
    if df is stats_reg_df:
        stat_cols = stat_cols_reg
    else:
        stat_cols = stat_cols_playoff
    # Further filter to only numeric columns
    numeric_stat_cols = df[stat_cols].select_dtypes(include=[np.number]).columns
    years_needed = [target_year - 2, target_year - 1, target_year]
    weighted = pd.Series(0.0, index=numeric_stat_cols)
    for w, y in zip(weights, years_needed):
        sub = df[(df['Player'] == player) & (df['year'] == y)]
        if sub.empty:
            # If any year missing, return all NaN
            return pd.Series([np.nan] * len(numeric_stat_cols), index=numeric_stat_cols)
        # Extract only numeric stat columns
        weighted += w * sub[numeric_stat_cols].iloc[0]
    return weighted

In [29]:
# Prepare list to hold enriched contract rows
enriched_rows = []

for _, row in contract_df.iterrows():
    player = row['Player']
    yr = int(row['year'])  # contract year
    yrs = row['YRS']
    retained = row['is_retained']
    
    # [修正重點 1]：把目標變數 Cap_Pct 抓出來
    cap_pct = row['Cap_Pct'] 

    # Get age from regular season stats for the contract year
    age_sub = stats_reg_df[(stats_reg_df['Player'] == player) & (stats_reg_df['year'] == yr)]
    age = age_sub['Age'].iloc[0] if not age_sub.empty else np.nan

    # Compute weighted regular stats
    wt_reg = weighted_stats(stats_reg_df, player, yr)

    # Compute weighted playoff stats
    wt_playoff = weighted_stats(stats_playoff_df, player, yr)

    # ==========================================
    # 策略一：處理季後賽遺失值與新增經驗旗標
    # ==========================================
    # 檢查是否所有的季後賽數據都是 NaN
    if wt_playoff.isna().all():
        has_playoff_exp = 0  # 沒打季後賽，設為 0
        wt_playoff = wt_reg.copy()  # 核心：用例行賽數據直接插補
    else:
        has_playoff_exp = 1  # 有打季後賽，設為 1

    # 插補完成後，再分別加上後綴區隔
    wt_reg = wt_reg.add_suffix('_reg')
    wt_playoff = wt_playoff.add_suffix('_playoff')
    # ==========================================

    # Build row
    new_row = {
        'Player': player,
        'YRS': yrs,
        'is_retained': retained,
        'year': yr,
        'age': age,
        'has_playoff_exp': has_playoff_exp,
        'Cap_Pct': cap_pct  # [修正重點 2]：把目標變數塞回字典裡！
    }
    
    # Add weighted stats
    for col in wt_reg.index:
        new_row[col] = wt_reg[col]
    for col in wt_playoff.index:
        new_row[col] = wt_playoff[col]

    enriched_rows.append(new_row)

# Create final dataframe
merged_df = pd.DataFrame(enriched_rows)

# [修正重點 3]：在重新排序欄位時，確保 Cap_Pct 在最前面（或最後面），方便後續切分 X 與 y
cols = ['Player', 'year', 'YRS', 'is_retained', 'age', 'has_playoff_exp', 'Cap_Pct'] \
        + sorted([c for c in merged_df.columns if c.endswith('_reg')]) \
        + sorted([c for c in merged_df.columns if c.endswith('_playoff')])

merged_df = merged_df[cols]

merged_df.head()

,Player,year,YRS,is_retained,age,has_playoff_exp,Cap_Pct,2P%_reg,2PA_reg,2P_reg,...,TOV%_playoff,TOV_playoff,TRB%_playoff,TRB_playoff,TS%_playoff,USG%_playoff,VORP_playoff,WS/48_playoff,WS_playoff,eFG%_playoff
0,stephen curry,2017,5,1,28.0,1,0.3500,0.5439,8.59,4.65,...,14.77,3.74,8.74,5.75,0.6318,30.83,1.76,0.2272,3.05,0.5797
1,blake griffin,2017,5,1,27.0,0,0.3000,0.5090,15.40,7.80,...,10.73,2.33,13.23,8.09,0.5579,28.62,3.04,0.1729,6.79,0.5089
2,gordon hayward,2017,4,0,26.0,0,0.3000,0.4923,10.32,5.07,...,11.21,2.24,8.54,5.18,0.5786,26.75,4.04,0.1754,9.61,0.5159
3,jrue holiday,2017,5,1,26.0,0,0.2592,0.4862,9.67,4.74,...,15.31,2.69,6.15,3.53,0.5291,25.04,1.79,0.0935,3.60,0.4991
4,otto porter,2017,4,0,23.0,0,0.2500,0.5470,5.48,2.99,...,7.12,0.71,10.17,5.36,0.5878,15.43,2.41,0.1406,6.92,0.5653


In [33]:
merged_df.dropna(inplace=True)
merged_df.to_csv('../../data/processed/merged_weighted_stats.csv', index=False)